# 🚀 BGE-M3 Fine-Tuning v2 — Pháp Luật Lao Động Việt Nam

Notebook tối ưu cho **Google Colab** (T4/L4/A100 GPU) — fine-tune mô hình `BAAI/bge-m3` trên bộ dữ liệu pháp luật lao động.

### ✨ Cải tiến so với v1

| Đặc điểm | v1 (cũ) | v2 (notebook này) |
|---|---|---|
| API | `model.fit()` (legacy) | `SentenceTransformerTrainer` (HuggingFace Trainer) |
| Dữ liệu | JSON tự parse | JSONL native, `datasets` library |
| Loss | MNRL đơn thuần | **MNRL + MatryoshkaLoss** (flexible dim) |
| Checkpoint | Save best cuối cùng | Auto checkpoint + resume training |
| Gradient | FP16 | **BF16** (A100/L4) hoặc FP16 (T4) + gradient checkpointing |
| Scheduler | Linear | **Cosine with warmup ratio** |
| Hard Negatives | 3 cố định | **Tất cả negatives** (5 per sample) |

### 📁 Dữ liệu cần upload lên Google Drive
```
MyDrive/
└── train_bgem3.jsonl   (~34MB, 2614 mẫu)
```

> **Hướng dẫn:** Upload file `train_bgem3.jsonl` vào thư mục gốc MyDrive trước khi chạy.

---
## 1. 🔍 Kiểm tra GPU & Cài đặt thư viện

In [ ]:
# Kiểm tra GPU được cấp
!nvidia-smi

import torch
print(f"\n🔧 PyTorch: {torch.__version__}")
print(f"🎮 GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB" if torch.cuda.is_available() else "")

In [ ]:
# Cài đặt thư viện — sentence-transformers >= 3.3 để dùng SentenceTransformerTrainer
!pip install -q -U \
    sentence-transformers>=3.3.0 \
    transformers>=4.41.0 \
    accelerate>=0.30.0 \
    datasets>=2.19.0

---
## 2. 📂 Kết nối Google Drive & Cấu hình đường dẫn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# ===== CẤU HÌNH ĐƯỜNG DẪN =====
TRAIN_FILE = Path('/content/drive/MyDrive/train_bgem3.jsonl')

# Thư mục output — lưu trên Drive để không mất khi Colab timeout
OUTPUT_DIR = Path('/content/drive/MyDrive/bge-m3-legal-v2')
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Kiểm tra dữ liệu
if TRAIN_FILE.exists():
    size_mb = TRAIN_FILE.stat().st_size / (1024*1024)
    print(f"✅ Đã tìm thấy: {TRAIN_FILE.name} ({size_mb:.2f} MB)")
else:
    print(f"❌ KHÔNG TÌM THẤY: {TRAIN_FILE}")
    print(f"   → Hãy upload file train_bgem3.jsonl vào thư mục gốc MyDrive!")

---
## 3. 📊 Load & Phân tích dữ liệu + Chia Train/Val

In [ ]:
import json
import random
import numpy as np
from collections import Counter

# Đọc toàn bộ dữ liệu
all_data = []
with open(TRAIN_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            all_data.append(json.loads(line))

print("=" * 60)
print("📊 THỐNG KÊ DỮ LIỆU")
print("=" * 60)
print(f"Tổng số mẫu: {len(all_data)}")

# Phân tích cấu trúc
has_negatives = sum(1 for d in all_data if d.get('negatives'))
neg_counts = [len(d.get('negatives', [])) for d in all_data]
anchor_lens = [len(d['anchor']) for d in all_data]
pos_lens = [len(d['positive']) for d in all_data]

print(f"Mẫu có hard negatives: {has_negatives}/{len(all_data)}")
print(f"Số negatives/mẫu — min: {min(neg_counts)}, max: {max(neg_counts)}, "
      f"median: {np.median(neg_counts):.0f}, mean: {np.mean(neg_counts):.1f}")
print(f"\nĐộ dài anchor (chars) — min: {min(anchor_lens)}, max: {max(anchor_lens)}, "
      f"median: {np.median(anchor_lens):.0f}")
print(f"Độ dài positive (chars) — min: {min(pos_lens)}, max: {max(pos_lens)}, "
      f"median: {np.median(pos_lens):.0f}")

type_dist = Counter(d.get('type', 'unknown') for d in all_data)
print(f"\nPhân bố type: {dict(type_dist)}")

# ===== CHIA TRAIN / VAL (90/10) =====
random.seed(42)
random.shuffle(all_data)

VAL_RATIO = 0.1
split_idx = int(len(all_data) * (1 - VAL_RATIO))
train_data = all_data[:split_idx]
val_data = all_data[split_idx:]

print(f"\n{'=' * 60}")
print(f"✅ Chia dữ liệu:")
print(f"   Train: {len(train_data)} mẫu (90%)")
print(f"   Val:   {len(val_data)} mẫu (10%)")

---
## 4. 🔄 Chuẩn bị Dataset cho SentenceTransformerTrainer

Sentence Transformers v3 sử dụng `datasets.Dataset` thay vì `InputExample`.

Với `MultipleNegativesRankingLoss`, format yêu cầu:
- Cột `anchor`: câu hỏi
- Cột `positive`: đoạn văn bản đúng
- Cột `negative_i` (0, 1, 2, ...): các hard negatives

In [ ]:
from datasets import Dataset

def prepare_dataset(data, max_negatives=5):
    """Chuyển đổi JSONL data thành datasets.Dataset format.

    Với MultipleNegativesRankingLoss:
    - Cột 'anchor': query/question
    - Cột 'positive': relevant passage
    - Cột 'negative_i': hard negative passages
    """
    records = []
    for item in data:
        anchor = item['anchor'].strip()
        positive = item['positive'].strip()

        if not anchor or not positive:
            continue

        record = {
            'anchor': anchor,
            'positive': positive,
        }

        # Thêm hard negatives
        negatives = item.get('negatives', [])
        for i, neg in enumerate(negatives[:max_negatives]):
            neg_text = neg.strip() if isinstance(neg, str) else neg
            record[f'negative_{i}'] = neg_text

        # Pad nếu thiếu negative columns (để tất cả records cùng schema)
        for i in range(len(negatives), max_negatives):
            record[f'negative_{i}'] = ""

        records.append(record)

    return Dataset.from_list(records)


# Xác định max_negatives từ dữ liệu
max_neg = max(len(d.get('negatives', [])) for d in all_data)
print(f"Số hard negatives tối đa trong dataset: {max_neg}")

# Dùng tất cả negatives có sẵn (tối đa 5)
MAX_NEGATIVES = min(max_neg, 5)
print(f"Sử dụng {MAX_NEGATIVES} hard negatives / mẫu")

train_dataset = prepare_dataset(train_data, max_negatives=MAX_NEGATIVES)
print(f"\n✅ Train dataset: {len(train_dataset)} mẫu")
print(f"   Columns: {train_dataset.column_names}")
print(f"\n📝 Mẫu đầu tiên:")
sample = train_dataset[0]
print(f"   anchor:   {sample['anchor'][:100]}...")
print(f"   positive: {sample['positive'][:100]}...")
for i in range(MAX_NEGATIVES):
    neg = sample.get(f'negative_{i}', '')
    if neg:
        print(f"   neg_{i}:    {neg[:80]}...")

---
## 5. 📐 Chuẩn bị Evaluator từ tập Validation

In [ ]:
from sentence_transformers.evaluation import InformationRetrievalEvaluator

def build_ir_evaluator(val_data):
    """Xây dựng InformationRetrievalEvaluator từ tập validation.

    Mỗi mẫu val có anchor/positive/negatives → dùng positive làm relevant,
    negatives + positives từ mẫu khác làm corpus.
    """
    queries = {}
    corpus = {}
    relevant_docs = {}

    doc_idx = 0

    for i, item in enumerate(val_data):
        qid = f'q_{i}'
        anchor = item['anchor'].strip()
        positive = item['positive'].strip()

        if not anchor or not positive:
            continue

        queries[qid] = anchor
        relevant_docs[qid] = set()

        # Positive document → relevant
        pos_doc_id = f'doc_{doc_idx}'
        doc_idx += 1
        corpus[pos_doc_id] = positive
        relevant_docs[qid].add(pos_doc_id)

        # Negative documents → not relevant (nhưng thêm vào corpus)
        for neg in item.get('negatives', []):
            neg_text = neg.strip() if isinstance(neg, str) else neg
            if neg_text:
                neg_doc_id = f'doc_{doc_idx}'
                doc_idx += 1
                corpus[neg_doc_id] = neg_text

    print(f"Queries: {len(queries)}")
    print(f"Corpus:  {len(corpus)} documents")
    avg_rel = np.mean([len(v) for v in relevant_docs.values()])
    print(f"Relevant docs/query: {avg_rel:.1f}")

    evaluator = InformationRetrievalEvaluator(
        queries=queries,
        corpus=corpus,
        relevant_docs=relevant_docs,
        name='val_legal_retrieval',
        mrr_at_k=[1, 3, 5, 10],
        ndcg_at_k=[3, 5, 10],
        accuracy_at_k=[1, 3, 5, 10],
        precision_recall_at_k=[5, 10],
        show_progress_bar=True,
        batch_size=64,
    )

    return evaluator

evaluator = build_ir_evaluator(val_data)
print("\n✅ Evaluator sẵn sàng!")

---
## 6. 🤖 Load mô hình BGE-M3 & Cấu hình Loss

### Chiến lược Loss: MatryoshkaLoss + MultipleNegativesRankingLoss

- **`MultipleNegativesRankingLoss`**: SOTA loss cho dense retrieval — dùng cả hard negatives lẫn in-batch negatives.
- **`MatryoshkaLoss`**: Cho phép tạo embeddings ở nhiều chiều (1024, 768, 512, 256) mà vẫn hiệu quả — rất hữu ích khi cần giảm dimensionality cho production.

In [ ]:
from sentence_transformers import SentenceTransformer, losses

MODEL_NAME = 'BAAI/bge-m3'

print(f"🔄 Đang tải mô hình {MODEL_NAME}...")
model = SentenceTransformer(MODEL_NAME)

# Đặt max_seq_length — 512 phủ >98% văn bản pháp luật, an toàn cho T4
model.max_seq_length = 512
print(f"📏 Max sequence length: {model.max_seq_length}")
print(f"📐 Embedding dimension: {model.get_sentence_embedding_dimension()}")
print(f"🔢 Tổng parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

# === Loss Configuration ===
# Inner loss: MultipleNegativesRankingLoss
inner_loss = losses.MultipleNegativesRankingLoss(model=model)

# Outer loss: MatryoshkaLoss — cho phép truncate embedding dimensions
matryoshka_dims = [1024, 768, 512, 256]
train_loss = losses.MatryoshkaLoss(
    model=model,
    loss=inner_loss,
    matryoshka_dims=matryoshka_dims,
)

print(f"\n✅ Loss: MatryoshkaLoss(MNRL)")
print(f"   Matryoshka dims: {matryoshka_dims}")
print(f"   In-batch negatives: Enabled")
print(f"   Hard negatives per sample: {MAX_NEGATIVES}")

---
## 7. ⚙️ Cấu hình siêu tham số & Training Arguments

### Giải thích chiến lược siêu tham số

| Tham số | Giá trị | Lý do |
|---|---|---|
| **Epochs** | 3 | BGE-M3 đã pretrained rất tốt, 3 epochs đủ hội tụ mà không overfit |
| **Batch size** | 8 | Phù hợp T4 15GB VRAM với 512 max_len + 5 negatives |
| **Grad accumulation** | 4 | Effective batch = 32 — tăng chất lượng in-batch negatives |
| **Learning rate** | 2e-5 | Conservative LR cho fine-tuning embedding model lớn |
| **Warmup ratio** | 0.1 | 10% steps đầu warmup, tránh catastrophic forgetting |
| **Scheduler** | Cosine | Giảm LR mượt mà, tốt hơn linear cho embedding models |
| **Weight decay** | 0.01 | Regularization nhẹ chống overfitting |
| **Grad checkpointing** | True | Tiết kiệm ~30% VRAM, trade-off ~15% tốc độ |

In [ ]:
from sentence_transformers import SentenceTransformerTrainingArguments, SentenceTransformerTrainer
import math

# ===== SIÊU THAM SỐ =====
NUM_EPOCHS = 3
BATCH_SIZE = 8              # Per-device batch size
GRAD_ACCUM_STEPS = 4        # Effective batch size = 8 * 4 = 32
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1          # 10% warmup
WEIGHT_DECAY = 0.01

# Tính toán số steps
steps_per_epoch = math.ceil(len(train_dataset) / (BATCH_SIZE * GRAD_ACCUM_STEPS))
total_steps = steps_per_epoch * NUM_EPOCHS
eval_steps = max(steps_per_epoch // 2, 50)  # Eval 2 lần mỗi epoch
save_steps = eval_steps  # Save cùng lúc eval

print("=" * 60)
print("⚙️  CẤU HÌNH HUẤN LUYỆN")
print("=" * 60)
print(f"Epochs:               {NUM_EPOCHS}")
print(f"Per-device batch:     {BATCH_SIZE}")
print(f"Gradient accumulation:{GRAD_ACCUM_STEPS}")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"Steps per epoch:      {steps_per_epoch}")
print(f"Total steps:          {total_steps}")
print(f"Warmup steps:         {int(total_steps * WARMUP_RATIO)}")
print(f"Eval every:           {eval_steps} steps")
print(f"Learning rate:        {LEARNING_RATE}")
print(f"Weight decay:         {WEIGHT_DECAY}")
print(f"Scheduler:            cosine")

# Detect GPU type cho precision setting
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ""
use_bf16 = any(g in gpu_name for g in ['A100', 'L4', 'H100', 'A10'])
use_fp16 = not use_bf16 and torch.cuda.is_available()
print(f"\nPrecision:            {'BF16' if use_bf16 else 'FP16' if use_fp16 else 'FP32'}")
print(f"GPU detected:         {gpu_name}")

# Training Arguments
training_args = SentenceTransformerTrainingArguments(
    output_dir=str(CHECKPOINT_DIR),

    # === Training ===
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    gradient_checkpointing=True,  # Tiết kiệm VRAM

    # === Optimizer ===
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type='cosine',
    warmup_ratio=WARMUP_RATIO,
    optim='adamw_torch_fused',  # Faster than default adamw

    # === Mixed Precision ===
    fp16=use_fp16,
    bf16=use_bf16,

    # === Evaluation & Saving ===
    eval_strategy='steps',
    eval_steps=eval_steps,
    save_strategy='steps',
    save_steps=save_steps,
    save_total_limit=3,             # Giữ tối đa 3 checkpoints (tiết kiệm Drive)
    load_best_model_at_end=True,
    metric_for_best_model='val_legal_retrieval_ndcg@10',
    greater_is_better=True,

    # === Logging ===
    logging_steps=20,
    logging_first_step=True,
    report_to='none',  # Đổi thành 'wandb' nếu muốn track trên Weights & Biases

    # === Misc ===
    seed=42,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    remove_unused_columns=False,
    batch_sampler='no_duplicates',  # Tránh duplicate trong batch → tốt hơn cho MNRL
)

print("\n✅ Training Arguments đã cấu hình xong!")

---
## 8. 🏋️ Huấn luyện

In [ ]:
# Khởi tạo Trainer
trainer = SentenceTransformerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator,
)

print("=" * 65)
print("🏋️ BẮT ĐẦU HUẤN LUYỆN BGE-M3 v2")
print("=" * 65)

# Train!
train_result = trainer.train()

print("\n" + "=" * 65)
print("✅ HUẤN LUYỆN HOÀN TẤT!")
print("=" * 65)
print(f"Total steps:     {train_result.global_step}")
print(f"Training loss:   {train_result.training_loss:.4f}")
print(f"Training time:   {train_result.metrics.get('train_runtime', 0):.0f}s")

---
## 9. 💾 Lưu mô hình tốt nhất vào Google Drive

In [ ]:
# Lưu mô hình cuối cùng (best model đã được load ở bước trên)
FINAL_MODEL_DIR = OUTPUT_DIR / 'final'
model.save(str(FINAL_MODEL_DIR))

print(f"✅ Mô hình đã lưu tại: {FINAL_MODEL_DIR}")

# Kiểm tra kích thước
total_size = sum(
    f.stat().st_size for f in FINAL_MODEL_DIR.rglob('*') if f.is_file()
) / (1024**3)
print(f"📦 Kích thước: {total_size:.2f} GB")

# Liệt kê files đã lưu
print("\n📁 Files:")
for f in sorted(FINAL_MODEL_DIR.rglob('*')):
    if f.is_file():
        size = f.stat().st_size / (1024**2)
        print(f"   {f.relative_to(FINAL_MODEL_DIR)}: {size:.1f} MB")

---
## 10. 📈 Đánh giá cuối cùng & So sánh với mô hình gốc

In [ ]:
# Đánh giá mô hình gốc (baseline)
print("📊 Đánh giá mô hình GỐC (BAAI/bge-m3)...")
base_model = SentenceTransformer('BAAI/bge-m3')
base_model.max_seq_length = 512
base_results = evaluator(base_model)

# Đánh giá mô hình fine-tuned
print("\n📊 Đánh giá mô hình FINE-TUNED...")
ft_model = SentenceTransformer(str(FINAL_MODEL_DIR))
ft_results = evaluator(ft_model)

# So sánh kết quả
print("\n" + "=" * 75)
print("📊 SO SÁNH KẾT QUẢ")
print("=" * 75)
print(f"{'Metric':<45} {'Gốc':>10} {'Fine-tuned':>10} {'Δ':>10}")
print("-" * 75)

key_metrics = [
    'val_legal_retrieval_ndcg@10',
    'val_legal_retrieval_ndcg@5',
    'val_legal_retrieval_ndcg@3',
    'val_legal_retrieval_mrr@10',
    'val_legal_retrieval_mrr@5',
    'val_legal_retrieval_mrr@3',
    'val_legal_retrieval_mrr@1',
    'val_legal_retrieval_accuracy@1',
    'val_legal_retrieval_accuracy@3',
    'val_legal_retrieval_accuracy@5',
    'val_legal_retrieval_accuracy@10',
    'val_legal_retrieval_precision@5',
    'val_legal_retrieval_precision@10',
    'val_legal_retrieval_recall@5',
    'val_legal_retrieval_recall@10',
]

for metric in key_metrics:
    base_val = base_results.get(metric, 0)
    ft_val = ft_results.get(metric, 0)
    delta = ft_val - base_val
    arrow = '⬆️' if delta > 0.001 else '⬇️' if delta < -0.001 else '➡️'
    print(f"{metric:<45} {base_val:>9.4f} {ft_val:>10.4f} {arrow}{abs(delta):>8.4f}")

---
## 11. 🧪 Kiểm tra trực quan: So sánh Cosine Similarity

In [ ]:
# Test queries thực tế về luật lao động
test_cases = [
    {
        'query': 'Người lao động thử việc có được trả lương không và mức lương tối thiểu là bao nhiêu?',
        'docs': [
            'Điều 26. Tiền lương thử việc: Tiền lương của người lao động trong thời gian thử việc do hai bên thỏa thuận nhưng ít nhất phải bằng 85% mức lương của công việc đó.',
            'Điều 90. Tiền lương: Tiền lương là số tiền mà người sử dụng lao động trả cho người lao động theo thỏa thuận để thực hiện công việc.',
            'Điều 120. Phân định thẩm quyền xử phạt: Chủ tịch UBND các cấp có quyền xử phạt vi phạm hành chính trong lĩnh vực giao thông.',
        ],
    },
    {
        'query': 'Các trường hợp chấm dứt hưởng trợ cấp thất nghiệp',
        'docs': [
            'Điều 53. Tạm dừng, tiếp tục, chấm dứt hưởng trợ cấp thất nghiệp: Người đang hưởng trợ cấp thất nghiệp bị chấm dứt hưởng trợ cấp thất nghiệp khi hết thời hạn, tìm được việc làm, hưởng lương hưu.',
            'Điều 49. Điều kiện hưởng: Người lao động đang đóng bảo hiểm thất nghiệp được hưởng trợ cấp thất nghiệp khi chấm dứt hợp đồng lao động.',
            'Điều 10. Nguyên tắc bảo vệ rừng: Quản lý rừng bền vững, bảo tồn đa dạng sinh học và cảnh quan thiên nhiên.',
        ],
    },
]

for i, case in enumerate(test_cases, 1):
    query = case['query']
    docs = case['docs']

    print(f"\n{'=' * 80}")
    print(f"🔍 Test {i}: \"{query}\"")
    print(f"{'=' * 80}")

    # Encode với cả 2 mô hình
    q_base = base_model.encode(query, normalize_embeddings=True)
    d_base = base_model.encode(docs, normalize_embeddings=True)
    scores_base = d_base @ q_base

    q_ft = ft_model.encode(query, normalize_embeddings=True)
    d_ft = ft_model.encode(docs, normalize_embeddings=True)
    scores_ft = d_ft @ q_ft

    print(f"\n{'Doc':<65} {'Gốc':>8} {'FT':>8} {'Δ':>8}")
    print("-" * 95)
    for doc, s_b, s_f in zip(docs, scores_base, scores_ft):
        delta = s_f - s_b
        print(f"{doc[:63]:<65} {s_b:>7.4f} {s_f:>8.4f} {delta:>+7.4f}")

---
## 12. 🔬 (Tùy chọn) Test Matryoshka — Hiệu năng ở các dimensions thấp

In [ ]:
# Test hiệu năng ở các Matryoshka dimensions
# Hữu ích khi cần giảm kích thước embedding cho production

print("🔬 Đánh giá hiệu năng Matryoshka ở các dimensions:")
print("=" * 60)

for dim in matryoshka_dims:
    # Tạo model với truncated dimensions
    ft_model_trunc = SentenceTransformer(str(FINAL_MODEL_DIR), truncate_dim=dim)
    results = evaluator(ft_model_trunc)

    ndcg10 = results.get('val_legal_retrieval_ndcg@10', 0)
    mrr10 = results.get('val_legal_retrieval_mrr@10', 0)
    acc1 = results.get('val_legal_retrieval_accuracy@1', 0)

    print(f"  Dim={dim:>4}: NDCG@10={ndcg10:.4f} | MRR@10={mrr10:.4f} | Acc@1={acc1:.4f}")

    del ft_model_trunc
    torch.cuda.empty_cache()

print("\n✅ Matryoshka evaluation hoàn tất!")
print("💡 Tip: Nếu dim=512 hoặc 256 vẫn cho kết quả tốt,")
print("   bạn có thể dùng dimension thấp hơn để tiết kiệm bộ nhớ Qdrant!")

---
## 📥 Hướng Dẫn Sử Dụng Mô Hình Đã Fine-Tune

### Trên Colab / Server:
```python
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('/content/drive/MyDrive/bge-m3-legal-v2/final')

# Hoặc dùng Matryoshka dim thấp hơn cho production
model = SentenceTransformer('...', truncate_dim=512)  

embeddings = model.encode(['Câu hỏi pháp luật...'], normalize_embeddings=True)
```

### Tích hợp vào PBL6:
1. Tải thư mục `bge-m3-legal-v2/final` từ Google Drive về máy
2. Đặt vào: `d:\PBL6_v2\PBL6\data\models\bge-m3-legal-v2`
3. Cập nhật `.env`:
   ```env
   EMBEDDING_MODEL=d:/PBL6_v2/PBL6/data/models/bge-m3-legal-v2
   ```
4. Re-embed và re-index collection Qdrant!